TASK 1.3

In [0]:
spark.sql("DROP TABLE IF EXISTS discussion_forum_bronze")

In [0]:
from pyspark.sql.functions import *
import pandas as pd

In [0]:
discussion_path = "/Volumes/edtech_project/edtech_bronze/edtech_project/discussion_posts/"

# Read NDJSON (line-delimited JSON)
discussion_df = (spark.read
    .format("json")
    .option("multiline", False)  # each line is a JSON object
    .load(discussion_path)
)

print("Row count after read:", discussion_df.count())  

# Store raw JSON for flexibility
discussion_df = discussion_df.withColumn("raw_json", to_json(struct([col(c) for c in discussion_df.columns])))

# Add metadata columns
discussion_df = (discussion_df
    .withColumn("_source_file", col("_metadata.file_path"))   # if working in your setup
    .withColumn("_load_ts", current_timestamp())
    .withColumn("_schema_version", lit("v1"))
    .withColumn("_last_modified_ts", current_timestamp())
)

In [0]:
(discussion_df.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("post_ts")  # or to_date("post_ts") if you want daily partitions
    .saveAsTable("discussion_forum_bronze")
)

In [0]:

%sql
-- display the table
SELECT * 
FROM discussion_forum_bronze
limit 5;


In [0]:

%sql
--display partitions
SHOW PARTITIONS discussion_forum_bronze;

In [0]:
%sql
select count(*) from discussion_forum_bronze;